In [ ]:
import os
import copy
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef, precision_score
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
class PairedDataset(Dataset):
    def __init__(self, pairs, transform_leaf, transform_fruit, class_to_idx):
        self.pairs = pairs
        self.transform_leaf = transform_leaf
        self.transform_fruit = transform_fruit
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        leaf_path, fruit_path, label_str = self.pairs[idx]
        leaf_img = Image.open(leaf_path).convert('RGB')
        fruit_img = Image.open(fruit_path).convert('RGB')

        leaf_img = self.transform_leaf(leaf_img)
        fruit_img = self.transform_fruit(fruit_img)
        label = self.class_to_idx[label_str]
        return leaf_img, fruit_img, label

def get_aligned_image_pairs(leaf_dir, fruit_dir):
    pairs = []
    for cls in sorted(os.listdir(leaf_dir)):
        leaf_imgs = sorted(os.listdir(os.path.join(leaf_dir, cls)))
        fruit_imgs = sorted(os.listdir(os.path.join(fruit_dir, cls)))
        for i in range(min(len(leaf_imgs), len(fruit_imgs))):
            pairs.append((
                os.path.join(leaf_dir, cls, leaf_imgs[i]),
                os.path.join(fruit_dir, cls, fruit_imgs[i]),
                cls
            ))
    return pairs


In [ ]:
def get_model(name, num_classes=4):
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "resnet50":
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "convnext_tiny":
        model = models.convnext_tiny(pretrained=True)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    else:
        raise ValueError("Unsupported model")
    return model

In [ ]:
class ModalityAttention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.attn_fc = nn.Sequential(
            nn.Linear(num_classes * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=1)
        )

    def forward(self, leaf_logits, fruit_logits):
        x = torch.cat([leaf_logits, fruit_logits], dim=1)
        weights = self.attn_fc(x)
        leaf_w, fruit_w = weights[:, 0].unsqueeze(1), weights[:, 1].unsqueeze(1)
        fused = leaf_w * leaf_logits + fruit_w * fruit_logits
        return fused, weights

class FusionModel(nn.Module):
    def __init__(self, leaf_model, fruit_model, num_classes=4):
        super().__init__()
        self.leaf_model = leaf_model
        self.fruit_model = fruit_model
        self.attn = ModalityAttention(num_classes)

    def forward(self, leaf, fruit):
        leaf_logits = self.leaf_model(leaf)
        fruit_logits = self.fruit_model(fruit)
        fused, attn_weights = self.attn(leaf_logits, fruit_logits)
        return fused, attn_weights


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef
)

def train_kfold_fusion_model(
    leaf_dir,
    fruit_dir,
    model_type="efficientnet_b0",
    num_classes=4,
    k_folds=5,
    num_epochs=10,
    batch_size=32,
    lr=1e-4,
    seed=42
):
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])

    pairs = get_aligned_image_pairs(leaf_dir, fruit_dir)
    class_names = sorted(list(set([lbl for _, _, lbl in pairs])))
    class_to_idx = {cls: i for i, cls in enumerate(class_names)}
    targets = [class_to_idx[label] for _, _, label in pairs]

    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(pairs, targets)):
        print(f"\n🟢 Fold {fold+1}/{k_folds}")

        train_data = [pairs[i] for i in train_idx]
        val_data = [pairs[i] for i in val_idx]

        train_ds = PairedDataset(train_data, transform, transform, class_to_idx)
        val_ds = PairedDataset(val_data, transform, transform, class_to_idx)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        leaf_model = get_model(model_type, num_classes).to(device)
        fruit_model = get_model(model_type, num_classes).to(device)
        fusion_model = FusionModel(leaf_model, fruit_model, num_classes).to(device)

        optimizer = torch.optim.Adam(fusion_model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0
        best_weights = copy.deepcopy(fusion_model.state_dict())

        for epoch in range(num_epochs):
            print(f"📚 Epoch {epoch+1}")
            fusion_model.train()
            for leaf_imgs, fruit_imgs, labels in tqdm(train_loader):
                leaf_imgs, fruit_imgs, labels = leaf_imgs.to(device), fruit_imgs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs, _ = fusion_model(leaf_imgs, fruit_imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            fusion_model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for leaf_imgs, fruit_imgs, labels in val_loader:
                    leaf_imgs, fruit_imgs = leaf_imgs.to(device), fruit_imgs.to(device)
                    labels = labels.to(device)
                    outputs, _ = fusion_model(leaf_imgs, fruit_imgs)
                    preds = outputs.argmax(dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            acc = accuracy_score(all_labels, all_preds)
            f1 = f1_score(all_labels, all_preds, average='macro')
            prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
            rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
            ber = 1 - balanced_accuracy_score(all_labels, all_preds)
            kappa = cohen_kappa_score(all_labels, all_preds)
            mcc = matthews_corrcoef(all_labels, all_preds)

            print(f"✅ Fold {fold+1} Epoch {epoch+1} — Acc: {acc:.4f}, F1: {f1:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, BER: {ber:.4f}")

            if acc > best_val_acc:
                best_val_acc = acc
                best_weights = copy.deepcopy(fusion_model.state_dict())

        torch.save(best_weights, f"fusion_model_{model_type}_fold{fold+1}.pth")

        results.append({
            "Fold": fold+1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "BER": ber,
            "Kappa": kappa,
            "MCC": mcc
        })

    df = pd.DataFrame(results)
    print("\n📈 K-Fold Summary:")
    print(df)

    print("\n📊 Mean ± Std:")
    for metric in ["Accuracy", "Precision", "Recall", "F1 Score", "BER", "Kappa", "MCC"]:
        print(f"{metric}: {df[metric].mean():.4f} ± {df[metric].std():.4f}")

    return df


In [ ]:
train_kfold_fusion_model(
    leaf_dir="/content/drive/MyDrive/datasets/Split_Leaves/train",
    fruit_dir="/content/drive/MyDrive/datasets/Mango/Split_Fruits/train",
    model_type="efficientnet_b0",
    num_classes=4,
    k_folds=5,
    num_epochs=10,
    batch_size=32,
    lr=1e-4
)



🟢 Fold 1/5


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


📚 Epoch 1


100%|██████████| 26/26 [00:39<00:00,  1.50s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9171, F1: 0.8862, Prec: 0.9234, Rec: 0.8810, BER: 0.1190
📚 Epoch 2


100%|██████████| 26/26 [00:38<00:00,  1.48s/it]


... (102 outputs left out)

✅ Fold 5 Epoch 9 — Acc: 0.9951, F1: 0.9936, Prec: 0.9924, Rec: 0.9950, BER: 0.0050
📚 Epoch 10


100%|██████████| 26/26 [00:40<00:00,  1.55s/it]


✅ Fold 5 Epoch 10 — Acc: 0.9951, F1: 0.9936, Prec: 0.9924, Rec: 0.9950, BER: 0.0050

📈 K-Fold Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.990244   0.994382  0.990000  0.992057  0.010000  0.986134  0.986271
1     2  0.980488   0.972973  0.980000  0.975298  0.020000  0.972400  0.972925
2     3  0.995122   0.993056  0.995098  0.994004  0.004902  0.993079  0.993113
3     4  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
4     5  0.995098   0.992424  0.995000  0.993629  0.005000  0.993041  0.993075

📊 Mean ± Std:
Accuracy: 0.9922 ± 0.0074
Precision: 0.9906 ± 0.0103
Recall: 0.9920 ± 0.0076
F1 Score: 0.9910 ± 0.0093
BER: 0.0080 ± 0.0076
Kappa: 0.9889 ± 0.0105
MCC: 0.9891 ± 0.0103


,Fold,Accuracy,Precision,Recall,F1 Score,BER,Kappa,MCC
0,1,0.990244,0.994382,0.990000,0.992057,0.010000,0.986134,0.986271
1,2,0.980488,0.972973,0.980000,0.975298,0.020000,0.972400,0.972925
2,3,0.995122,0.993056,0.995098,0.994004,0.004902,0.993079,0.993113
3,4,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
4,5,0.995098,0.992424,0.995000,0.993629,0.005000,0.993041,0.993075


In [ ]:
train_kfold_fusion_model(
    leaf_dir="/content/drive/MyDrive/datasets/Split_Leaves/train",
    fruit_dir="/content/drive/MyDrive/datasets/Mango/Split_Fruits/train",
    model_type="resnet50",
    num_classes=4,
    k_folds=5,
    num_epochs=10,
    batch_size=32,
    lr=1e-4
)



🟢 Fold 1/5


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


📚 Epoch 1


100%|██████████| 26/26 [00:40<00:00,  1.56s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9951, F1: 0.9947, Prec: 0.9972, Rec: 0.9924, BER: 0.0076
📚 Epoch 2


100%|██████████| 26/26 [00:39<00:00,  1.54s/it]


... (102 outputs left out)

✅ Fold 5 Epoch 9 — Acc: 0.9902, F1: 0.9899, Prec: 0.9902, Rec: 0.9900, BER: 0.0100
📚 Epoch 10


100%|██████████| 26/26 [00:41<00:00,  1.60s/it]


✅ Fold 5 Epoch 10 — Acc: 1.0000, F1: 1.0000, Prec: 1.0000, Rec: 1.0000, BER: 0.0000

📈 K-Fold Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.995122   0.995098  0.997126  0.996080  0.002874  0.993093  0.993127
1     2  0.990244   0.994382  0.987424  0.990788  0.012576  0.986126  0.986231
2     3  0.995122   0.997159  0.992188  0.994603  0.007812  0.993063  0.993098
3     4  0.985294   0.978571  0.978571  0.977612  0.021429  0.979118  0.979418
4     5  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000

📊 Mean ± Std:
Accuracy: 0.9932 ± 0.0056
Precision: 0.9930 ± 0.0084
Recall: 0.9911 ± 0.0085
F1 Score: 0.9918 ± 0.0086
BER: 0.0089 ± 0.0085
Kappa: 0.9903 ± 0.0079
MCC: 0.9904 ± 0.0078


,Fold,Accuracy,Precision,Recall,F1 Score,BER,Kappa,MCC
0,1,0.995122,0.995098,0.997126,0.996080,0.002874,0.993093,0.993127
1,2,0.990244,0.994382,0.987424,0.990788,0.012576,0.986126,0.986231
2,3,0.995122,0.997159,0.992188,0.994603,0.007812,0.993063,0.993098
3,4,0.985294,0.978571,0.978571,0.977612,0.021429,0.979118,0.979418
4,5,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000


In [ ]:
train_kfold_fusion_model(
    leaf_dir="/content/drive/MyDrive/datasets/Split_Leaves/train",
    fruit_dir="/content/drive/MyDrive/datasets/Mango/Split_Fruits/train",
    model_type="mobilenet_v2",
    num_classes=4,
    k_folds=5,
    num_epochs=10,
    batch_size=32,
    lr=1e-4
)



🟢 Fold 1/5


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 128MB/s]


📚 Epoch 1


100%|██████████| 26/26 [00:36<00:00,  1.42s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9707, F1: 0.9652, Prec: 0.9733, Rec: 0.9592, BER: 0.0408
📚 Epoch 2


100%|██████████| 26/26 [00:36<00:00,  1.41s/it]


... (102 outputs left out)

✅ Fold 5 Epoch 9 — Acc: 1.0000, F1: 1.0000, Prec: 1.0000, Rec: 1.0000, BER: 0.0000
📚 Epoch 10


100%|██████████| 26/26 [00:38<00:00,  1.47s/it]


✅ Fold 5 Epoch 10 — Acc: 1.0000, F1: 1.0000, Prec: 1.0000, Rec: 1.0000, BER: 0.0000

📈 K-Fold Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
1     2  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
2     3  0.995122   0.993056  0.995098  0.994004  0.004902  0.993079  0.993113
3     4  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
4     5  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000

📊 Mean ± Std:
Accuracy: 0.9990 ± 0.0022
Precision: 0.9986 ± 0.0031
Recall: 0.9990 ± 0.0022
F1 Score: 0.9988 ± 0.0027
BER: 0.0010 ± 0.0022
Kappa: 0.9986 ± 0.0031
MCC: 0.9986 ± 0.0031


,Fold,Accuracy,Precision,Recall,F1 Score,BER,Kappa,MCC
0,1,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
1,2,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
2,3,0.995122,0.993056,0.995098,0.994004,0.004902,0.993079,0.993113
3,4,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
4,5,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000


In [ ]:
train_kfold_fusion_model(
    leaf_dir="/content/drive/MyDrive/datasets/Split_Leaves/train",
    fruit_dir="/content/drive/MyDrive/datasets/Mango/Split_Fruits/train",
    model_type="convnext_tiny",
    num_classes=4,
    k_folds=5,
    num_epochs=10,
    batch_size=32,
    lr=1e-4
)



🟢 Fold 1/5


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


📚 Epoch 1


100%|██████████| 26/26 [00:57<00:00,  2.20s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9854, F1: 0.9854, Prec: 0.9917, Rec: 0.9798, BER: 0.0202
📚 Epoch 2


100%|██████████| 26/26 [00:57<00:00,  2.19s/it]


... (102 outputs left out)

✅ Fold 5 Epoch 9 — Acc: 1.0000, F1: 1.0000, Prec: 1.0000, Rec: 1.0000, BER: 0.0000
📚 Epoch 10


100%|██████████| 26/26 [00:59<00:00,  2.27s/it]


✅ Fold 5 Epoch 10 — Acc: 1.0000, F1: 1.0000, Prec: 1.0000, Rec: 1.0000, BER: 0.0000

📈 K-Fold Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
1     2  0.995122   0.992647  0.995000  0.993743  0.005000  0.993088  0.993122
2     3  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
3     4  0.995098   0.993056  0.992188  0.992511  0.007812  0.993036  0.993070
4     5  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000

📊 Mean ± Std:
Accuracy: 0.9980 ± 0.0027
Precision: 0.9971 ± 0.0039
Recall: 0.9974 ± 0.0036
F1 Score: 0.9973 ± 0.0038
BER: 0.0026 ± 0.0036
Kappa: 0.9972 ± 0.0038
MCC: 0.9972 ± 0.0038


,Fold,Accuracy,Precision,Recall,F1 Score,BER,Kappa,MCC
0,1,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
1,2,0.995122,0.992647,0.995000,0.993743,0.005000,0.993088,0.993122
2,3,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
3,4,0.995098,0.993056,0.992188,0.992511,0.007812,0.993036,0.993070
4,5,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
